# 02. 特徴量重要度分析
LightGBMで勝利予測モデルを学習し、SHAP値で各ファクターの寄与度を可視化する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import sys; sys.path.append('../src')
from evaluate_roi import roi_from_model_proba

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features.csv', encoding='utf-8-sig')
df = df.sort_values('race_id').reset_index(drop=True)
print(f'shape: {df.shape}, races: {df["race_id"].nunique()}')

In [ ]:
# 特徴量リスト（リーク列は除外）
FEATURES = [
    # オッズ系
    'log_odds', 'odds_rank_pct', 'prob_norm', '単勝人気',
    'log_odds_gap', 'odds_gap',
    # 予想・得点系
    '得点', 'デフォルト得点', '得点V1', '得点V2', '得点V3',
    '予想タイム指数', '予想タイム指数回帰推定値', 'time_idx_gap',
    '得点_rank_pct', '予想タイム指数_rank_pct',
    # 評価系
    '騎手評価', '調教師評価', '枠順評価', '脚質評価',
    'jockey_trainer_score',
    # 先行・展開系
    '先行指数', '先行率', '予想展開',
    # 血統
    '血統距離評価', '血統トラック評価', '血統成長力評価', '血統総合評価',
    # 馬属性
    '馬齢', 'sex_code', '馬体重', '馬体重増減', 'weight_abs_diff',
    'weight_up', 'weight_down', '負担重量', '斤量比',
    # コース
    '距離', 'log_distance', 'is_turf', 'is_dirt',
    'is_sprint', 'is_mile', 'is_middle', 'is_long',
    'cond_code', '天候コード',
    # 前走情報
    '前走着順', '前走人気', '前走着差', '前走馬体重', '前走頭数',
    '休養週数', '休養後出走回数', '距離増減',
    # レース特性
    '波乱度', 'レースレベル', '頭数', '競走種別コード',
    # その他
    'キャリア', '過去5走最高タイム指数', '前走レースレベル',
]
# データに存在する列のみ使用
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET = 'label_win'

X = df[FEATURES].copy()
y = df[TARGET]
print(f'特徴量数: {len(FEATURES)}, 正例率: {y.mean():.3f}')

## 時系列クロスバリデーション

In [ ]:
# race_idの分位でfold分割（時系列順）
race_order = df['race_id'].rank(method='dense').astype(int)
n_races = df['race_id'].nunique()
fold_size = n_races // 5

oof_proba = np.zeros(len(df))
models = []
auc_scores = []

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 30,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'verbose': -1,
    'n_estimators': 1000,
    'early_stopping_rounds': 50,
}

race_ids_sorted = sorted(df['race_id'].unique())
for fold in range(5):
    val_races = set(race_ids_sorted[fold * fold_size: (fold + 1) * fold_size])
    train_races = set(race_ids_sorted[:fold * fold_size])
    if not train_races:
        continue
    train_mask = df['race_id'].isin(train_races)
    val_mask   = df['race_id'].isin(val_races)

    X_tr, X_val = X[train_mask], X[val_mask]
    y_tr, y_val = y[train_mask], y[val_mask]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.log_evaluation(period=200)])

    proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_mask.values] = proba
    auc = roc_auc_score(y_val, proba)
    auc_scores.append(auc)
    models.append(model)
    print(f'Fold {fold+1} AUC: {auc:.4f}')

print(f'\nOOF AUC: {np.mean(auc_scores):.4f} +/- {np.std(auc_scores):.4f}')

## LightGBM Feature Importance (Gain)

In [ ]:
fi_gain  = pd.Series(
    np.mean([m.booster_.feature_importance('gain') for m in models], axis=0),
    index=FEATURES).sort_values(ascending=False)
fi_split = pd.Series(
    np.mean([m.booster_.feature_importance('split') for m in models], axis=0),
    index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fi_gain.head(20).plot(kind='barh', ax=axes[0], title='Feature Importance (Gain) - Top 20')
fi_split.head(20).plot(kind='barh', ax=axes[1], title='Feature Importance (Split) - Top 20', color='orange')
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\n-- Top 20 Gain --')
print(fi_gain.head(20).to_string())

## SHAP値による寄与度分析

In [ ]:
last_model = models[-1]
val_races = set(race_ids_sorted[3 * fold_size: 4 * fold_size])
X_sample = X[df['race_id'].isin(val_races)]

explainer = shap.TreeExplainer(last_model)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 9))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False, max_display=20)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES,
                  plot_type='bar', show=False, max_display=20)
plt.title('SHAP Bar Plot (Mean |SHAP|)')
plt.tight_layout()
plt.show()

## SHAP 依存プロット（重要上位特徴量）

In [ ]:
top4 = fi_gain.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flat, top4):
    feat_idx = list(FEATURES).index(feat)
    ax.scatter(X_sample[feat], shap_values[:, feat_idx], alpha=0.3, s=8)
    ax.set_xlabel(feat)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'{feat} SHAP Dependence')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

## OOF予測確率を保存

In [ ]:
df['pred_win_proba'] = oof_proba
df.to_csv('../data/processed/features_with_pred.csv', index=False, encoding='utf-8-sig')
print('保存完了: data/processed/features_with_pred.csv')
print(f'OOFカバレッジ: {(df["pred_win_proba"] > 0).sum()} / {len(df)} 行')